# Patrón de Diseño ETL / ELT en Databricks PySpark

Este notebook estructurado contiene el flujo estándar de desarrollo de procesos ETL en PySpark/Databricks, organizado por secciones con sus respectivas descripciones y bloques de código.

## 1. Cabecera
> **Descripción:** Contará con información general del proceso facilitando su entendimiento, así como también mantendrá la lista de cambios o modificaciones aplicadas a este.

In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Data Warehouse Comercial - Clientes
# PROCESO        : ETL_CLIENTES
# OBJETIVO       : Consolidar información de clientes
# VERSION        : 1.2.0
# DESARROLLADOR  : Eduardo Fajardo
# FECHA          : 04/09/2026
# TABLA FUENTE   : mb_silver_prod.mmff.m_cliente_stg
# TABLA DESTINO  : mb_gold_prod.comercial.dim_cliente
# FRECUENCIA     : Diaria
# -------------------------------------------------------------------------

## 2. Importación de librerías
> **Descripción:** Contará con las librerías y funciones necesarias para la ejecución de la lógica de negocio propuesta.

In [ ]:
import logging
import time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 3. Lectura de parámetros
> **Descripción:** Contará con los parámetros de ejecución requeridos para cada ambiente, de acuerdo a la necesidad de ellos.

In [ ]:
dbutils.widgets.text("p_catalogo", "")
dbutils.widgets.text("p_esquema", "")
dbutils.widgets.text("p_fecha_proceso", "")

var_catalogo = dbutils.widgets.get("p_catalogo")
var_esquema = dbutils.widgets.get("p_esquema")
var_fecha_proceso = dbutils.widgets.get("p_fecha_proceso")

logger = logging.getLogger("ETL_CLIENTES")
logger.setLevel(logging.INFO)

ini_proceso = time.perf_counter()
logger.info("Inicio del proceso ETL_CLIENTES")
logger.info("Parametros de los widgets: catalogo=%s esquema=%s fecha=%s",
            var_catalogo, var_esquema, var_fecha_proceso)

print("Parametros cargados correctamente")

## 4. Sección constantes
> **Descripción:** Contará con los valores que necesitan ser constantes a lo largo del proceso.

In [ ]:
TBL_CLIENTES_SRC = f"{var_catalogo}.{var_esquema}.m_cliente_stg"
TBL_CLIENTES_FIN = f"{var_catalogo}.{var_esquema}.dim_cliente"
TBL_LOG_PROCESOS = f"{var_catalogo}.{var_esquema}.log_ejecucion"

COLUMNAS_ORIGEN = [
    "cod_cliente",
    "nom_cliente",
    "tip_documento",
    "est_cliente",
    "fec_proceso",
    "fec_actualizacion",
]

LLAVE_DEDUPLICACION = "cod_cliente"
FORMATO_FECHA = "yyyy-MM-dd"

## 5. Funciones de transformación del proceso
> **Descripción:** Contará con funciones modularizadas de Lectura/Transformación/Escritura que cumplan con la función principal del proceso. Cada una de ellas representará un paso dentro del ETL.

In [ ]:
def read_clientes(tabla, columnas):
    """Lee los clientes desde el origen."""
    return spark.table(tabla).select(*columnas)


def add_nombre_normalizado(df_origen):
    """Normaliza el nombre del cliente a mayusculas y sin espacios."""
    return df_origen.withColumn(
        "nom_cliente", F.upper(F.trim(F.col("nom_cliente")))
    )


def add_fecha_carga(df_origen, fecha_proceso):
    """Agrega la fecha de proceso como marca de carga."""
    return df_origen.withColumn(
        "fec_carga", F.to_date(F.lit(fecha_proceso), FORMATO_FECHA)
    )


def add_antiguedad_cliente(df_origen):
    """Calcula los dias transcurridos desde la ultima actualizacion."""
    return df_origen.withColumn(
        "ctd_dias_antiguedad",
        F.datediff(F.current_date(), F.col("fec_actualizacion"))
    )

## 6. Lógica del proceso
> **Descripción:** Actuará como orquestador de las funciones definidas previamente. Establecerá el flujo de ejecución del ETL de forma general.

In [ ]:
ini_etapa = time.perf_counter()

fecha_corte = "2026-01-01"
estado_cliente = "ACTIVO"

df_clientes = read_clientes(TBL_CLIENTES_SRC, COLUMNAS_ORIGEN)

df_transformado = (
    df_clientes
    .transform(add_nombre_normalizado)
    .transform(lambda df_paso: add_fecha_carga(df_paso, var_fecha_proceso))
    .transform(add_antiguedad_cliente)
    .filter(F.col("fec_proceso") >= fecha_corte)
    .filter(F.col("est_cliente") == estado_cliente)
)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 7. Reglas de precarga
> **Descripción:** Se aplicarán reglas de precarga a valores descriptivos/informativos, con la finalidad de cumplir con recomendaciones de gobierno.

In [ ]:
def validate_reglas_precarga(df_origen, codigo_col, descriptivo_col):
    """Marca DATO NO INFORMADO o FUERA DE DOMINIO segun el codigo."""
    sin_codigo = (
        F.col(codigo_col).isNull() | (F.trim(F.col(codigo_col)) == "")
    )
    sin_descripcion = (
        F.col(descriptivo_col).isNull()
        | (F.trim(F.col(descriptivo_col)) == "")
    )
    return df_origen.withColumn(
        descriptivo_col,
        F.when(sin_codigo, F.lit("DATO NO INFORMADO"))
        .when(
            F.col(codigo_col).isNotNull() & sin_descripcion,
            F.lit("FUERA DE DOMINIO")
        )
        .otherwise(F.col(descriptivo_col))
    )


df_precargado = validate_reglas_precarga(
    df_transformado, "tip_documento", "nom_cliente"
)

## 8. Rejectados (Eliminación de duplicados)
> **Descripción:** Se aplicará una lógica de deduplicación para prevenir el ingreso de valores duplicados a la tabla final, de acuerdo a las llaves establecidas previamente por cada tabla.

In [ ]:
ventana_cliente = Window.partitionBy(LLAVE_DEDUPLICACION).orderBy(
    F.col("fec_actualizacion").desc()
)

df_sin_duplicados = (
    df_precargado
    .withColumn("nro_orden", F.row_number().over(ventana_cliente))
    .filter(F.col("nro_orden") == 1)
    .drop("nro_orden")
)

## 9. Inserción a tabla final
> **Descripción:** Se insertarán los registros en la tabla final del proceso de acuerdo a configuraciones/particiones previamente establecidas.

In [ ]:
ini_escritura = time.perf_counter()

try:
    (
        df_sin_duplicados
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_CLIENTES_FIN)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_CLIENTES_FIN, exc)
    raise

logger.info("Tiempo de escritura: %.2f segundos",
            time.perf_counter() - ini_escritura)

## 10. Inserción a tabla de ejecución proceso
> **Descripción:** Se creará un registro del tiempo, proceso, estado en la tabla de ejecuciones de los procesos.

In [ ]:
ctd_registros = df_sin_duplicados.count()

df_log = spark.createDataFrame(
    [("ETL_CLIENTES", var_fecha_proceso, ctd_registros, datetime.now(), "OK")],
    ["des_proceso", "fec_proceso", "ctd_registro", "fec_ejecucion", "est_ejecucion"],
)

try:
    df_log.write.format("delta").mode("append").saveAsTable(TBL_LOG_PROCESOS)
except Exception as exc:
    logger.error("Error al escribir la tabla de ejecucion %s: %s",
                 TBL_LOG_PROCESOS, exc)

print("Proceso ETL_CLIENTES finalizado")

logger.info("Fin del proceso ETL_CLIENTES. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)